PART 1 - SIMPLE AGENT AND AGENT LOOP

In [8]:
import os 
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override = True)

from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI

In [12]:
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

nvidia_client = AsyncOpenAI(base_url=NVIDIA_BASE_URL, api_key=NVIDIA_API_KEY)

nvidia_model = OpenAIChatCompletionsModel(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    openai_client = nvidia_client
)

In [13]:
agent = Agent(
            name = 'Jokester',
            instructions = 'You are joke teller',
            model=nvidia_model
            )

In [16]:
result = await Runner.run(agent, "Bana bir şaka yap")

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [17]:
print(result.final_output)

Temel, düğüne giderken yolda bir arkadaşıyla karşılaşır.
Arkadaşı: "Nereye gidiyorsun Temel?"
Temel: "Dügüne gidiyorum."
Arkadaşı: "Peki, ne götürüyorsun hediye?"
Temel: "Electrikli süpürge götürüyorum."
Arkadaşı (şaşkınlıkla): "Elektrikli süpürge mi? Düğüne mi gider bu?"
Temel: "Evet, dediler ki 'gelin beyaz eşya istiyor', ben de süpürgeyi aldım!"


In [18]:
result.to_input_list()

[{'content': 'Bana bir şaka yap', 'role': 'user'},
 {'id': '__fake_id__',
  'summary': [{'text': 'The user wants a joke in Turkish.',
    'type': 'summary_text'}],
  'type': 'reasoning',
  'provider_data': {'model': 'nvidia/nemotron-3-ultra-550b-a55b',
   'response_id': 'chatcmpl-93dd9685-468d-4b5b-baa9-2310cbe37fc4'}},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Temel, düğüne giderken yolda bir arkadaşıyla karşılaşır.\nArkadaşı: "Nereye gidiyorsun Temel?"\nTemel: "Dügüne gidiyorum."\nArkadaşı: "Peki, ne götürüyorsun hediye?"\nTemel: "Electrikli süpürge götürüyorum."\nArkadaşı (şaşkınlıkla): "Elektrikli süpürge mi? Düğüne mi gider bu?"\nTemel: "Evet, dediler ki \'gelin beyaz eşya istiyor\', ben de süpürgeyi aldım!"',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'nvidia/nemotron-3-ultra-550b-a55b',
   'response_id': 'chatcmpl-93dd9685-468d-4b5b-baa9-2310cbe37fc

In [19]:
with trace("Şaka yapıyor"):
    result = await Runner.run(agent, "Bana bir şaka yap")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Temel, hastaneye gidiyor.
Doktor: "Temel bey, size iki haberim var. Biri kötü, diğeri daha da kötü."
Temel: "Önce kötüyü söyleyin doktor bey."
Doktor: "Kötü haber: Size ömrünüz 24 saat kala."
Temel (şok olmuş): "Aman Allah'ım! Peki ya daha kötü haber ne?"
Doktor: "Daha kötü haber: Dün size söylemeye çalıştım ama telefonu açmadınız."


OPENAI_API_KEY is not set, skipping trace export


PART 2 - ADDING A TOOL

In [20]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [24]:
@function_tool
def push_tool(message: str) -> str:
    "Send this to user as a push notification"
    payload = {'user': pushover_user, 'token': pushover_token, 'message': message}
    results = requests.post(pushover_url, data=payload).status_code
    return f'Push sent with API status code {result}'

In [25]:
push_tool

FunctionTool(name='push_tool', description='Send this to user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7fba5d51b820>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [26]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [27]:
notifier_agent = Agent(
    name = 'Notifier',
    instructions = 'You are a push notification sender. You will send push notifications to the user',
    tools = [push_tool],
    model=nvidia_model
)

In [29]:
with trace('Pizza has arrived'):
    result = await Runner.run(notifier_agent, "Notify the user that their pizza has arrived")
print(result.final_output)

Your pizza arrival notification has been sent! 🍕


OPENAI_API_KEY is not set, skipping trace export


PART 3 - MEMORY

In [30]:
agent = Agent(
    name="Assistant",
    model=nvidia_model)

In [33]:
reponse = await Runner.run(agent, "Merhaba ben Muhammed.")
print(reponse.final_output)

Merhaba Muhammed! Hoş geldin. Bugün sana nasıl yardımcı olabilirim?


OPENAI_API_KEY is not set, skipping trace export


In [37]:
reponse.to_input_list()

[{'content': 'Merhaba ben Muhammed.', 'role': 'user'},
 {'id': '__fake_id__',
  'summary': [{'text': 'The user said "Merhaba ben Muhammed." (Hello, I am Muhammed).\nThis is a standard greeting in Turkish.\nI should reply in Turkish, greeting them back and asking how I can help.',
    'type': 'summary_text'}],
  'type': 'reasoning',
  'provider_data': {'model': 'nvidia/nemotron-3-ultra-550b-a55b',
   'response_id': 'chatcmpl-c314f6f5-ad49-4174-bf79-7864a86bb54b'}},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Merhaba Muhammed! Hoş geldin. Bugün sana nasıl yardımcı olabilirim?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'nvidia/nemotron-3-ultra-550b-a55b',
   'response_id': 'chatcmpl-c314f6f5-ad49-4174-bf79-7864a86bb54b'}}]

In [38]:
next_input = reponse.to_input_list() + [{"role": "user", "content": "Benim adım ne?"}]
next_input

[{'content': 'Merhaba ben Muhammed.', 'role': 'user'},
 {'id': '__fake_id__',
  'summary': [{'text': 'The user said "Merhaba ben Muhammed." (Hello, I am Muhammed).\nThis is a standard greeting in Turkish.\nI should reply in Turkish, greeting them back and asking how I can help.',
    'type': 'summary_text'}],
  'type': 'reasoning',
  'provider_data': {'model': 'nvidia/nemotron-3-ultra-550b-a55b',
   'response_id': 'chatcmpl-c314f6f5-ad49-4174-bf79-7864a86bb54b'}},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Merhaba Muhammed! Hoş geldin. Bugün sana nasıl yardımcı olabilirim?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'nvidia/nemotron-3-ultra-550b-a55b',
   'response_id': 'chatcmpl-c314f6f5-ad49-4174-bf79-7864a86bb54b'}},
 {'role': 'user', 'content': 'Benim adım ne?'}]

In [39]:
reponse = await Runner.run(agent, next_input)
print(reponse.final_output)

OPENAI_API_KEY is not set, skipping trace export


Adın **Muhammed**. Az önce "Merhaba ben Muhammed" dediğini hatırlıyorum. 😊


OPENAI_API_KEY is not set, skipping trace export


SQLITE SESSION APPROACH

In [40]:
session = SQLiteSession("12345")

In [41]:
response = await Runner.run(agent, "Merhaba ben Muhammed.", session=session)
print(response.final_output)

Merhaba Muhammed! Nasıl yardımcı olabilirim sana bugün? 😊


OPENAI_API_KEY is not set, skipping trace export


In [42]:
response = await Runner.run(agent, 'Merhaba, benim adım ne?', session=session)
print(response.final_output)

Adın **Muhammed**. Bir önceki mesajında "Merhaba ben Muhammed" diye tanıtmıştın kendini. 😊


OPENAI_API_KEY is not set, skipping trace export
